# 🏏 Multi-Format Cricket Analytics
### ODI | Test | T20I | IPL | PSL

### Step 1 — Download

In [ ]:
!wget -q https://cricsheet.org/downloads/odis_csv2.zip && !unzip -q odis_csv2.zip -d odi_data/
!wget -q https://cricsheet.org/downloads/tests_csv2.zip && !unzip -q tests_csv2.zip -d test_data/
!wget -q https://cricsheet.org/downloads/t20s_csv2.zip && !unzip -q t20s_csv2.zip -d t20i_data/
!wget -q https://cricsheet.org/downloads/ipl_csv2.zip  && !unzip -q ipl_csv2.zip  -d ipl_data/
!wget -q https://cricsheet.org/downloads/psl_csv2.zip  && !unzip -q psl_csv2.zip  -d psl_data/
print("✓ Done")

### Step 2 — Load & Tag

In [ ]:
import os, pandas as pd

def load_format(folder, label):
    files = [f for f in os.listdir(folder) if not f.endswith('_info.csv') and f.endswith('.csv')]
    dfs = []
    for f in sorted(files):
        try: dfs.append(pd.read_csv(f'{folder}/{f}'))
        except: pass
    df = pd.concat(dfs, ignore_index=True)
    df['format'] = label
    print(f"✓ {label:6s} {df.shape[0]:,} rows")
    return df

df = pd.concat([
    load_format('odi_data/','ODI'), load_format('test_data/','Test'),
    load_format('t20i_data/','T20I'), load_format('ipl_data/','IPL'),
    load_format('psl_data/','PSL')
], ignore_index=True)
print(f"\n✓ TOTAL {df.shape[0]:,} rows")

### Step 3 — Clean & Validate

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Dupes removed: {before-len(df):,}")

required = ['match_id','striker','bowler','runs_off_bat','ball','start_date']
df = df.dropna(subset=required)
df = df[df['runs_off_bat'].between(0,6)]

for col in ['wides','noballs','byes','legbyes','extras']:
    df[col] = pd.to_numeric(df.get(col, 0), errors='coerce').fillna(0)

df['bowler_runs'] = df['runs_off_bat'] + df['wides'] + df['noballs']
df['start_date']  = pd.to_datetime(df['start_date'], errors='coerce')
df = df.dropna(subset=['start_date'])
df['over']      = df['ball'].astype(str).str.split('.').str[0].astype(int)
df['total_runs']= df['runs_off_bat'] + df['extras']
df['is_wicket'] = df['wicket_type'].notna().astype(int)
df['is_wide']   = (df['wides']>0).astype(int)
df['is_noball'] = (df['noballs']>0).astype(int)
df['is_dot']    = ((df['runs_off_bat']==0)&(df['is_wide']==0)&(df['is_noball']==0)).astype(int)
df['year']      = df['start_date'].dt.year

valid = df.groupby('match_id')['ball'].count()
df = df[df['match_id'].isin(valid[valid>=6].index)]
print(f"✓ CLEAN: {df.shape[0]:,} rows | {df['format'].value_counts().to_dict()}")

### Step 4 — Innings Table (for 50s, 100s, highest score, best bowling)

In [ ]:
# ── Batting innings ─────────────────────────────────────────────────────
bat_innings = df[df['is_wide']==0].groupby(
    ['match_id','striker','batting_team','bowling_team','venue','start_date','format']
).agg(
    runs        = ('runs_off_bat','sum'),
    balls_faced = ('runs_off_bat','count'),
    fours       = ('runs_off_bat', lambda x:(x==4).sum()),
    sixes       = ('runs_off_bat', lambda x:(x==6).sum()),
    dismissed   = ('is_wicket','max'),
).reset_index()
bat_innings['strike_rate'] = ((bat_innings['runs']/bat_innings['balls_faced'].replace(0,1))*100).round(2)
bat_innings['year'] = pd.to_datetime(bat_innings['start_date']).dt.year

# ── Bowling innings ──────────────────────────────────────────────────────
bowl_innings = df[df['is_wide']==0].groupby(
    ['match_id','bowler','batting_team','bowling_team','venue','start_date','format']
).agg(
    balls      = ('bowler_runs','count'),
    runs_given = ('bowler_runs','sum'),
    wickets    = ('is_wicket','sum'),
    dot_balls  = ('is_dot','sum'),
).reset_index()
bowl_innings['overs']   = (bowl_innings['balls']/6).round(1)
bowl_innings['economy'] = ((bowl_innings['runs_given']/bowl_innings['balls'].replace(0,1))*6).round(2)
bowl_innings['year']    = pd.to_datetime(bowl_innings['start_date']).dt.year

print(f"✓ bat_innings: {bat_innings.shape} | bowl_innings: {bowl_innings.shape}")

### Step 5 — Career Stats + Milestones

In [ ]:
def build_batting(data, group_cols=['striker']):
    bat = data.groupby(group_cols).agg(
        matches      = ('match_id','nunique'),
        runs         = ('runs_off_bat','sum'),
        balls_faced  = ('is_wide', lambda x:(x==0).sum()),
        dismissals   = ('is_wicket','sum'),
        dot_balls    = ('is_dot','sum'),
        fours        = ('runs_off_bat', lambda x:(x==4).sum()),
        sixes        = ('runs_off_bat', lambda x:(x==6).sum()),
    ).reset_index()
    bat['average']      = (bat['runs']/bat['dismissals'].replace(0,1)).round(2)
    bat['strike_rate']  = ((bat['runs']/bat['balls_faced'].replace(0,1))*100).round(2)
    bat['dot_pct']      = ((bat['dot_balls']/bat['balls_faced'].replace(0,1))*100).round(2)
    bat['boundary_pct'] = (((bat['fours']+bat['sixes'])/bat['balls_faced'].replace(0,1))*100).round(2)
    return bat.sort_values('runs',ascending=False)

def build_bowling(data, group_cols=['bowler']):
    bowl = data.groupby(group_cols).agg(
        matches    = ('match_id','nunique'),
        balls      = ('is_wide', lambda x:(x==0).sum()),
        runs_given = ('bowler_runs','sum'),
        wickets    = ('is_wicket','sum'),
        dot_balls  = ('is_dot','sum'),
        wides      = ('is_wide','sum'),
        noballs    = ('is_noball','sum'),
    ).reset_index()
    bowl['overs']       = (bowl['balls']/6).round(1)
    bowl['economy']     = ((bowl['runs_given']/bowl['balls'].replace(0,1))*6).round(2)
    bowl['average']     = (bowl['runs_given']/bowl['wickets'].replace(0,1)).round(2)
    bowl['dot_pct']     = ((bowl['dot_balls']/bowl['balls'].replace(0,1))*100).round(2)
    bowl['strike_rate'] = (bowl['balls']/bowl['wickets'].replace(0,1)).round(2)
    return bowl.sort_values('wickets',ascending=False)

batting           = build_batting(df)
bowling           = build_bowling(df)
batting_by_format = build_batting(df,['striker','format'])
bowling_by_format = build_bowling(df,['bowler','format'])

# ── Milestones from innings table ────────────────────────────────────────
bat_milestones = bat_innings.groupby(['striker','format']).agg(
    hundreds    = ('runs', lambda x:(x>=100).sum()),
    fifties     = ('runs', lambda x:((x>=50)&(x<100)).sum()),
    thirties    = ('runs', lambda x:((x>=30)&(x<50)).sum()),
    highest     = ('runs','max'),
    ducks       = ('runs', lambda x:(x==0).sum()),
).reset_index()

bowl_milestones = bowl_innings.groupby(['bowler','format']).agg(
    five_wkts   = ('wickets', lambda x:(x>=5).sum()),
    four_wkts   = ('wickets', lambda x:(x==4).sum()),
    best_wkts   = ('wickets','max'),
).reset_index()
# best figures = most wickets, then fewest runs in that spell
best_fig = bowl_innings.sort_values(['wickets','runs_given'],ascending=[False,True])
best_fig = best_fig.groupby(['bowler','format']).first()[['wickets','runs_given']].reset_index()
best_fig['best_bowling'] = best_fig['wickets'].astype(str)+'/'+best_fig['runs_given'].astype(str)
bowl_milestones = bowl_milestones.merge(best_fig[['bowler','format','best_bowling']],on=['bowler','format'],how='left')

batting_by_format = batting_by_format.merge(bat_milestones, on=['striker','format'], how='left')
bowling_by_format = bowling_by_format.merge(bowl_milestones, on=['bowler','format'], how='left')

print(f"✓ batting_by_format: {batting_by_format.shape}")
print(f"✓ bowling_by_format: {bowling_by_format.shape}")
print("Sample Kohli ODI:", batting_by_format[
    (batting_by_format['striker'].str.contains('Kohli',na=False))&
    (batting_by_format['format']=='ODI')
][['striker','runs','hundreds','fifties','highest']].to_string())

### Step 6 — Yearly, Venue, Opponent, Matchup Tables

In [ ]:
batting_yearly = df.groupby(['striker','year','format']).agg(
    runs=('runs_off_bat','sum'), balls_faced=('is_wide',lambda x:(x==0).sum()),
    dismissals=('is_wicket','sum'), matches=('match_id','nunique'),
    fours=('runs_off_bat',lambda x:(x==4).sum()), sixes=('runs_off_bat',lambda x:(x==6).sum()),
).reset_index()
batting_yearly['average']     = (batting_yearly['runs']/batting_yearly['dismissals'].replace(0,1)).round(2)
batting_yearly['strike_rate'] = ((batting_yearly['runs']/batting_yearly['balls_faced'].replace(0,1))*100).round(2)

bowling_yearly = df.groupby(['bowler','year','format']).agg(
    balls=('is_wide',lambda x:(x==0).sum()), runs_given=('bowler_runs','sum'),
    wickets=('is_wicket','sum'), matches=('match_id','nunique'), dot_balls=('is_dot','sum'),
).reset_index()
bowling_yearly['economy']     = ((bowling_yearly['runs_given']/bowling_yearly['balls'].replace(0,1))*6).round(2)
bowling_yearly['average']     = (bowling_yearly['runs_given']/bowling_yearly['wickets'].replace(0,1)).round(2)
bowling_yearly['strike_rate'] = (bowling_yearly['balls']/bowling_yearly['wickets'].replace(0,1)).round(2)

batting_venue = df.groupby(['striker','venue','format']).agg(
    innings=('match_id','nunique'), runs=('runs_off_bat','sum'),
    balls_faced=('is_wide',lambda x:(x==0).sum()), dismissals=('is_wicket','sum'),
    fours=('runs_off_bat',lambda x:(x==4).sum()), sixes=('runs_off_bat',lambda x:(x==6).sum()),
).reset_index()
batting_venue['average']     = (batting_venue['runs']/batting_venue['dismissals'].replace(0,1)).round(2)
batting_venue['strike_rate'] = ((batting_venue['runs']/batting_venue['balls_faced'].replace(0,1))*100).round(2)

batting_opponent = df.groupby(['striker','bowling_team','format']).agg(
    innings=('match_id','nunique'), runs=('runs_off_bat','sum'),
    balls_faced=('is_wide',lambda x:(x==0).sum()), dismissals=('is_wicket','sum'),
    fours=('runs_off_bat',lambda x:(x==4).sum()), sixes=('runs_off_bat',lambda x:(x==6).sum()),
).reset_index()
batting_opponent['average']     = (batting_opponent['runs']/batting_opponent['dismissals'].replace(0,1)).round(2)
batting_opponent['strike_rate'] = ((batting_opponent['runs']/batting_opponent['balls_faced'].replace(0,1))*100).round(2)
batting_opponent.rename(columns={'bowling_team':'opponent'},inplace=True)

bowling_venue = df.groupby(['bowler','venue','format']).agg(
    innings=('match_id','nunique'), balls=('is_wide',lambda x:(x==0).sum()),
    runs_given=('bowler_runs','sum'), wickets=('is_wicket','sum'), dot_balls=('is_dot','sum'),
).reset_index()
bowling_venue['economy'] = ((bowling_venue['runs_given']/bowling_venue['balls'].replace(0,1))*6).round(2)
bowling_venue['average'] = (bowling_venue['runs_given']/bowling_venue['wickets'].replace(0,1)).round(2)
bowling_venue['dot_pct'] = ((bowling_venue['dot_balls']/bowling_venue['balls'].replace(0,1))*100).round(2)

bowling_opponent = df.groupby(['bowler','batting_team','format']).agg(
    innings=('match_id','nunique'), balls=('is_wide',lambda x:(x==0).sum()),
    runs_given=('bowler_runs','sum'), wickets=('is_wicket','sum'), dot_balls=('is_dot','sum'),
).reset_index()
bowling_opponent['economy'] = ((bowling_opponent['runs_given']/bowling_opponent['balls'].replace(0,1))*6).round(2)
bowling_opponent['average'] = (bowling_opponent['runs_given']/bowling_opponent['wickets'].replace(0,1)).round(2)
bowling_opponent['dot_pct'] = ((bowling_opponent['dot_balls']/bowling_opponent['balls'].replace(0,1))*100).round(2)
bowling_opponent.rename(columns={'batting_team':'opponent'},inplace=True)

batter_vs_bowler = df.groupby(['striker','bowler','format']).agg(
    balls_faced=('is_wide',lambda x:(x==0).sum()), runs=('runs_off_bat','sum'),
    dismissals=('is_wicket','sum'), fours=('runs_off_bat',lambda x:(x==4).sum()),
    sixes=('runs_off_bat',lambda x:(x==6).sum()), dot_balls=('is_dot','sum'),
).reset_index()
batter_vs_bowler['strike_rate'] = ((batter_vs_bowler['runs']/batter_vs_bowler['balls_faced'].replace(0,1))*100).round(2)
batter_vs_bowler['average']     = (batter_vs_bowler['runs']/batter_vs_bowler['dismissals'].replace(0,1)).round(2)
batter_vs_bowler = batter_vs_bowler[batter_vs_bowler['balls_faced']>=10]

bowler_vs_batter = df.groupby(['bowler','striker','format']).agg(
    balls_bowled=('is_wide',lambda x:(x==0).sum()), runs_given=('bowler_runs','sum'),
    wickets=('is_wicket','sum'), dot_balls=('is_dot','sum'),
    fours_given=('runs_off_bat',lambda x:(x==4).sum()),
    sixes_given=('runs_off_bat',lambda x:(x==6).sum()),
).reset_index()
bowler_vs_batter['economy']     = ((bowler_vs_batter['runs_given']/bowler_vs_batter['balls_bowled'].replace(0,1))*6).round(2)
bowler_vs_batter['strike_rate'] = (bowler_vs_batter['balls_bowled']/bowler_vs_batter['wickets'].replace(0,1)).round(2)
bowler_vs_batter['dot_pct']     = ((bowler_vs_batter['dot_balls']/bowler_vs_batter['balls_bowled'].replace(0,1))*100).round(2)
bowler_vs_batter = bowler_vs_batter[bowler_vs_batter['balls_bowled']>=10]
print("✓ All tables built")

### Step 7 — ML: Player Similarity + Form Rating + Player Score

In [ ]:
!pip install scikit-learn -q
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ── 1. Batting similarity ────────────────────────────────────────────────
bat_ml = batting_by_format.copy()
features = ['average','strike_rate','boundary_pct','dot_pct','runs']
bat_ml = bat_ml.dropna(subset=features)
bat_ml = bat_ml[bat_ml['runs'] >= 200]   # min threshold

scaler  = StandardScaler()
X       = scaler.fit_transform(bat_ml[features])
bat_ml['cluster'] = KMeans(n_clusters=8, random_state=42, n_init=10).fit_predict(X)

# cosine similarity matrix for finding similar players
bat_similarity_matrix = cosine_similarity(X)
bat_ml['_idx'] = range(len(bat_ml))

def similar_batters(name, fmt, top_n=10):
    mask = (bat_ml['striker'].str.contains(name,case=False,na=False)) & (bat_ml['format']==fmt)
    if mask.sum()==0: return pd.DataFrame()
    idx  = bat_ml[mask]['_idx'].iloc[0]
    sims = bat_similarity_matrix[idx]
    top  = np.argsort(sims)[::-1][1:top_n+1]
    result = bat_ml.iloc[top][['striker','format','runs','average','strike_rate','boundary_pct']].copy()
    result['similarity_%'] = (sims[top]*100).round(1)
    return result.reset_index(drop=True)

# ── 2. Bowling similarity ────────────────────────────────────────────────
bowl_ml = bowling_by_format.copy()
bfeatures = ['economy','average','dot_pct','strike_rate','wickets']
bowl_ml = bowl_ml.dropna(subset=bfeatures)
bowl_ml = bowl_ml[bowl_ml['wickets'] >= 20]

Xb      = scaler.fit_transform(bowl_ml[bfeatures])
bowl_ml['cluster'] = KMeans(n_clusters=8, random_state=42, n_init=10).fit_predict(Xb)
bowl_similarity_matrix = cosine_similarity(Xb)
bowl_ml['_idx'] = range(len(bowl_ml))

def similar_bowlers(name, fmt, top_n=10):
    mask = (bowl_ml['bowler'].str.contains(name,case=False,na=False)) & (bowl_ml['format']==fmt)
    if mask.sum()==0: return pd.DataFrame()
    idx  = bowl_ml[mask]['_idx'].iloc[0]
    sims = bowl_similarity_matrix[idx]
    top  = np.argsort(sims)[::-1][1:top_n+1]
    result = bowl_ml.iloc[top][['bowler','format','wickets','economy','average','dot_pct']].copy()
    result['similarity_%'] = (sims[top]*100).round(1)
    return result.reset_index(drop=True)

# ── 3. Form rating (last 2 years vs career) ──────────────────────────────
latest_year = batting_yearly['year'].max()
recent = batting_yearly[batting_yearly['year'] >= latest_year-1].groupby(['striker','format']).agg(
    recent_runs=('runs','sum'), recent_avg=('average','mean'), recent_sr=('strike_rate','mean')
).reset_index()
career_avg = batting_by_format[['striker','format','average','strike_rate']].copy()
career_avg.columns = ['striker','format','career_avg','career_sr']
form = recent.merge(career_avg, on=['striker','format'], how='left')
form['form_score'] = ((form['recent_avg']/form['career_avg'].replace(0,1))*50 +
                      (form['recent_sr']/form['career_sr'].replace(0,1))*50).round(1)
form['form_label'] = pd.cut(form['form_score'],
    bins=[0,60,85,110,999], labels=['📉 Poor','😐 Average','✅ Good','🔥 On Fire'])

# ── 4. Player Score (0-100) ───────────────────────────────────────────────
def compute_player_score(row):
    score = (
        min(row.get('average',0)/80,1)*30 +
        min(row.get('strike_rate',0)/180,1)*25 +
        min(row.get('boundary_pct',0)/30,1)*20 +
        min(row.get('runs',0)/10000,1)*15 +
        (1 - min(row.get('dot_pct',100)/70,1))*10
    )
    return round(score*100, 1)

batting_by_format['player_score'] = batting_by_format.apply(compute_player_score, axis=1)

print("✓ ML done — similarity, form ratings, player scores")
print("\nSample similar to Kohli ODI:")
print(similar_batters('Kohli','ODI',5))

### Step 8 — Save Everything

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

files_to_save = {
    'cricket_batting_stats.csv'    : batting,
    'cricket_bowling_stats.csv'    : bowling,
    'cricket_batting_by_format.csv': batting_by_format,   # now has 100s,50s,highest,player_score
    'cricket_bowling_by_format.csv': bowling_by_format,   # now has 5wkts,best_bowling
    'cricket_batting_yearly.csv'   : batting_yearly,
    'cricket_bowling_yearly.csv'   : bowling_yearly,
    'cricket_batting_venue.csv'    : batting_venue,
    'cricket_batting_opponent.csv' : batting_opponent,
    'cricket_bowling_venue.csv'    : bowling_venue,
    'cricket_bowling_opponent.csv' : bowling_opponent,
    'cricket_batter_vs_bowler.csv' : batter_vs_bowler,
    'cricket_bowler_vs_batter.csv' : bowler_vs_batter,
    'cricket_bat_innings.csv'      : bat_innings,
    'cricket_bowl_innings.csv'     : bowl_innings,
    'cricket_form_ratings.csv'     : form,
    'cricket_bat_similarity.csv'   : bat_ml[['striker','format','cluster','average','strike_rate','boundary_pct','dot_pct','runs','player_score']],
    'cricket_bowl_similarity.csv'  : bowl_ml[['bowler','format','cluster','wickets','economy','average','dot_pct']],
}

for fn, d in files_to_save.items():
    d.to_csv(f'/content/drive/MyDrive/{fn}', index=False)
    print(f"✓ {fn} — {d.shape[0]:,} rows")

### Step 9 — Download

In [ ]:
from google.colab import files
import shutil
for fn in files_to_save:
    shutil.copy(f'/content/drive/MyDrive/{fn}', f'/content/{fn}')
    files.download(f'/content/{fn}')
    print(f"↓ {fn}")